In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az

from bcr.data.download import download_coat
from bcr.data.preprocess import load_coat, make_synthetic_mnar
from bcr.models.bayesian_pmf import BayesianPMF
from bcr.evaluation.metrics import ndcg_at_k, recall_at_k, rmse_on_test

RANDOM_SEED = 42
sns.set_theme(style="whitegrid", palette="muted")

## Notebook 01 — Bayesian Probabilistic Matrix Factorization

### What is Bayesian PMF and why use it?

**Standard Matrix Factorization** decomposes the rating matrix R ≈ U Vᵀ and
returns point estimates for the latent user/item factors. It tells you *where*
the posterior is most likely, but not *how certain* you should be.

**Bayesian PMF** places priors over U and V and infers a full posterior
distribution. This gives us:

- **Calibrated uncertainty** — the model tells you which predictions it's
  confident about vs. which are wild guesses. Items with high posterior
  variance are candidates for exploration (Phase 3: Thompson Sampling).
- **Regularisation for free** — the prior σ_u, σ_v play the role of L2
  regularisation, but their strength is inferred from data.
- **Principled cold-start handling** — for users/items with few observations,
  the prior dominates and predictions shrink toward the global mean, rather
  than overfitting to noise.

### Model

```
sigma_u ~ HalfNormal(1)         # user factor scale
sigma_v ~ HalfNormal(1)         # item factor scale
tau     ~ HalfNormal(1)         # observation noise precision
U[u]    ~ Normal(0, sigma_u)    shape: (n_users, n_factors)
V[i]    ~ Normal(0, sigma_v)    shape: (n_items, n_factors)
R[u,i]  ~ Normal(U[u]·V[i], 1/tau)   for observed (u,i)
```

We use a **non-centred parameterisation** (U = U_offset × σ_u) to improve
NUTS mixing when σ is small — a common trick in hierarchical Bayesian models
(see McElreath 2020, Ch. 13).

Inference is via **NUTS** (No-U-Turn Sampler), giving asymptotically exact
posterior samples with minimal tuning.

In [2]:
# ── Load data ────────────────────────────────────────────────────────
USE_COAT = False
try:
    download_coat("data/raw")
    coat = load_coat("data/raw")
    full_train_r    = coat["train"]
    full_test_r     = coat["test"]
    full_train_mask = coat["train_mask"]
    full_test_mask  = coat["test_mask"]
    USE_COAT = True
    print(f"✓ Coat loaded — {full_train_r.shape[0]} users × {full_train_r.shape[1]} items")
except Exception as e:
    print(f"⚠️  Coat unavailable: {e}")
    print("Using synthetic MNAR dataset (500 users × 200 items).")
    syn = make_synthetic_mnar(n_users=500, n_items=200, n_factors=10, random_seed=RANDOM_SEED)
    full_train_r    = syn["train_ratings"]
    full_test_r     = syn["test_ratings"]
    full_train_mask = syn["train_mask"]
    full_test_mask  = syn["test_mask"]

dataset_name = "Coat" if USE_COAT else "Synthetic MNAR"
print(f"Dataset: {dataset_name} — {full_train_r.shape}")

[coat] Trying https://www.cs.cornell.edu/~schnabts/mnar/train.ascii ...


[coat]   → failed: 403 Client Error: Forbidden for url: https://www.cs.cornell.edu/~schnabts/mnar/train.ascii
[coat] Trying https://raw.githubusercontent.com/usaito/unbiased-implicit-rec-real/master/data/coat/train.ascii ...
[coat]   → failed: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/usaito/unbiased-implicit-rec-real/master/data/coat/train.ascii
⚠️  Coat unavailable: All mirrors failed for train.ascii.
Please download train.ascii and test.ascii manually from https://www.cs.cornell.edu/~schnabts/mnar/ and place them in data/raw/coat/
Using synthetic MNAR dataset (500 users × 200 items).
Dataset: Synthetic MNAR — (500, 200)


### 1 — Fit model on a small subset (fast iteration)

We first fit on the first **50 users**, all items, to iterate quickly on
diagnostics. Full-dataset fitting follows the same code.

In [3]:
N_USERS_SUBSET = 50
train_r    = full_train_r[:N_USERS_SUBSET]
train_mask = full_train_mask[:N_USERS_SUBSET]
test_r     = full_test_r[:N_USERS_SUBSET]
test_mask  = full_test_mask[:N_USERS_SUBSET]

print(f"Subset: {train_r.shape[0]} users × {train_r.shape[1]} items")
print(f"  Train observations: {train_mask.sum():,}")
print(f"  Test  observations: {test_mask.sum():,}")

Subset: 50 users × 200 items
  Train observations: 658
  Test  observations: 1,883


In [4]:
# Build and fit BayesianPMF
N_FACTORS = 10
model = BayesianPMF(n_factors=N_FACTORS, random_seed=RANDOM_SEED)
model.build_model(train_r, train_mask)

print(f"Model built: {train_mask.sum():,} observed ratings, {N_FACTORS} latent factors")
print("Starting NUTS sampling …")

trace = model.fit(draws=500, tune=500, chains=2, target_accept=0.9)
print("Sampling complete.")

Model built: 658 observed ratings, 10 latent factors
Starting NUTS sampling …


Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [sigma_u, sigma_v, tau, mu_global, sigma_bu, sigma_bi, bias_u_offset, bias_i_offset, U_offset, V_offset]


Output()

...

There were 39 divergences after tuning. Increase `target_accept` or reparameterize.


We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


⚠️  Convergence warning: R-hat ≥ 1.05 for tau. Consider increasing tune/draws.
Sampling complete.


### 2 — Convergence diagnostics

In [5]:
# ── R-hat summary ────────────────────────────────────────────────────
summary_global = az.summary(trace, var_names=["sigma_u", "sigma_v", "tau"])
print("Convergence summary (global parameters):")
print(summary_global.to_string())

max_rhat = summary_global["r_hat"].max()
if max_rhat < 1.05:
    print(f"\n✓ All R-hat < 1.05 (max = {max_rhat:.3f}) — chains converged.")
else:
    print(f"\n⚠️  Max R-hat = {max_rhat:.3f} — consider more tuning steps.")

Convergence summary (global parameters):
          mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
sigma_u  0.622  0.446   0.104    1.481      0.012    0.023    1595.0     650.0    1.0
sigma_v  0.623  0.448   0.104    1.482      0.012    0.020    1612.0     602.0    1.0
tau      2.818  0.426   2.053    3.590      0.096    0.037      25.0      35.0    1.1

⚠️  Max R-hat = 1.100 — consider more tuning steps.


In [6]:
# ── Trace plots ──────────────────────────────────────────────────────
ax = az.plot_trace(
    trace,
    var_names=["sigma_u", "sigma_v", "tau"],
    figsize=(12, 6),
    combined=False,
)
plt.suptitle("Trace plots — global hyperparameters", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("outputs/figures/01_trace_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [7]:
# ── Posterior plots ──────────────────────────────────────────────────
az.plot_posterior(
    trace,
    var_names=["sigma_u", "sigma_v", "tau"],
    figsize=(12, 4),
    kind="hist",
)
plt.suptitle("Posterior distributions — global hyperparameters", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("outputs/figures/01_posterior_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [8]:
# ── Energy plot ───────────────────────────────────────────────────────
az.plot_energy(trace, figsize=(8, 4))
plt.title("Energy plot — BFMI diagnostic")
plt.tight_layout()
plt.savefig("outputs/figures/01_energy_plot.png", dpi=150, bbox_inches="tight")
plt.show()

# BFMI < 0.2 is a warning sign
bfmi = az.bfmi(trace)
print(f"BFMI per chain: {bfmi}")
if any(b < 0.2 for b in bfmi):
    print("⚠️  Low BFMI — possible divergences; try more tuning steps or re-parameterisation.")
else:
    print("✓ BFMI looks healthy.")

BFMI per chain: [0.16458947 0.28006808]
⚠️  Low BFMI — possible divergences; try more tuning steps or re-parameterisation.


### 3 — Predictions and posterior predictive check

In [9]:
# ── Posterior predictive check ───────────────────────────────────────
with model.model:
    ppc = pm.sample_posterior_predictive(trace, random_seed=RANDOM_SEED, progressbar=False)

ppc_obs = ppc.posterior_predictive["obs"].values.reshape(-1)
actual_obs = train_r[train_mask]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(actual_obs, bins=20, alpha=0.6, label="Observed (train)", color="steelblue",
        density=True, edgecolor="white")
ax.hist(ppc_obs, bins=50, alpha=0.5, label="PPC samples", color="darkorange",
        density=True, edgecolor="none")
ax.set_xlabel("Rating value")
ax.set_ylabel("Density")
ax.set_title("Posterior Predictive Check — rating distribution")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/figures/01_ppc.png", dpi=150, bbox_inches="tight")
plt.show()

Sampling: [obs]


In [10]:
# ── Posterior mean ± 94% HDI for 3 sample users ──────────────────────
sample_users = [0, 1, 2]
TOP_ITEMS = 20

fig, axes = plt.subplots(len(sample_users), 1, figsize=(14, 4 * len(sample_users)))

for ax, u in zip(axes, sample_users):
    means, stds = model.predict_all_items(u)
    top_idx = np.argsort(means)[::-1][:TOP_ITEMS]

    # HDI bounds via approximate normal
    z94 = 1.88  # ≈ 94% CI half-width multiplier for normal
    lo = means[top_idx] - z94 * stds[top_idx]
    hi = means[top_idx] + z94 * stds[top_idx]

    xs = np.arange(TOP_ITEMS)
    ax.bar(xs, means[top_idx], color="steelblue", alpha=0.7, label="Posterior mean")
    ax.vlines(xs, lo, hi, color="black", linewidth=1.5, label="~94% HDI")

    # Mark items that appear in test set
    in_test = np.isin(top_idx, np.where(test_mask[u])[0])
    if in_test.any():
        ax.scatter(xs[in_test], means[top_idx][in_test] + 0.1,
                   marker="*", color="darkorange", zorder=5, s=120,
                   label="In test set")

    ax.set_title(f"User {u} — top-{TOP_ITEMS} items by posterior mean")
    ax.set_xlabel("Item rank")
    ax.set_ylabel("Predicted rating")
    ax.set_xticks(xs)
    ax.set_xticklabels(top_idx, rotation=45, fontsize=7)
    ax.legend(fontsize=8)

plt.suptitle("Posterior mean ± 94% HDI for sample users", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("outputs/figures/01_user_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

### 4 — Evaluation on unbiased test set

In [11]:
# ── Metrics ───────────────────────────────────────────────────────────
ndcg  = ndcg_at_k(model, test_r, test_mask, k=10)
rec   = recall_at_k(model, test_r, test_mask, k=10)
rmse  = rmse_on_test(model, test_r, test_mask)

print("=" * 45)
print("Results on unbiased test set:")
print(f"  NDCG@10  : {ndcg:.4f}")
print(f"  Recall@10: {rec:.4f}")
print(f"  RMSE     : {rmse:.4f}")
print("=" * 45)

Results on unbiased test set:
  NDCG@10  : 0.7082
  Recall@10: 0.0598
  RMSE     : 1.0340


In [12]:
# Save trace for Phase 2
import os
os.makedirs("outputs/traces", exist_ok=True)
az.to_netcdf(trace, "outputs/traces/bpmf_phase1.nc")
print("Trace saved to outputs/traces/bpmf_phase1.nc")

Trace saved to outputs/traces/bpmf_phase1.nc


### 5 — Reflection: what does posterior uncertainty tell us?

The posterior uncertainty captured in `std` and the 94% HDI has several
practical implications:

1. **Item popularity effect**: Items that appear rarely in the training data
   (long-tail items) have high posterior variance — the model is uncertain
   because it has seen little evidence. This is epistemically honest.

2. **User heterogeneity**: Some users have narrow HDI bands (they rated many
   items, constraining their latent factor U[u]); others have wide bands
   (new/sparse users). A point-estimate model hides this distinction.

3. **Exploration signal** (Phase 3 preview): The items with highest *uncertainty*
   are precisely the ones a Thompson Sampler should explore — if the true rating
   is high, we miss value by never recommending them; the posterior variance is
   the natural signal for exploration.

4. **MNAR caveat**: Even with calibrated uncertainty, the model was trained on
   *biased* data. Items that were never exposed to a user are absent from
   training — their latent factor V[i] is estimated only from other users'
   ratings. Phase 2 addresses this via causal debiasing.

---

**Note on convergence**: If R-hat ≥ 1.05 appeared above, the chains may not
have fully mixed. Common fixes:
- Increase `tune` from 500 to 1000+
- Increase `target_accept` to 0.95
- Reduce `n_factors` (fewer parameters to explore)
- The non-centred parameterisation in our model already helps; further
  improvement requires more data or better initialisation.